# Counting Communities (Group 20)

## 1. Introduction

...

In [1]:
# load libraries
import networkx as nx
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
import os, requests, zipfile, gzip, io
from scipy.stats import poisson
from types import SimpleNamespace
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

SEED = 2026

DATASETS = "data"
os.makedirs(DATASETS, exist_ok=True)

WEAVER_URL = "https://nrvis.com/download/data/dynamic/aves-weaver-social.zip"

def download(url, file_name):
    path = os.path.join(DATASETS, file_name)
    if not os.path.exists(path):
        print(f"downloading {file_name} ...")
        try:
            req = requests.get(url)
            req.raise_for_status() # in case of errors

            with open(path, 'wb') as f:
                f.write(req.content)

        except Exception as error:
            if os.path.exists(path):
                os.remove(path) # clean corrupted install
            raise RuntimeError(f"Graph download failed : {error}. If it continues to fail, download them by hand and save them in a 'data' folder in the root directory")

    return path

def csv_path(name):
    return os.path.join(DATASETS, name)

def save_edges(G, file_name):
    nx.to_pandas_edgelist(G)[["source", "target"]].to_csv(csv_path(file_name), index=False)

def read_edges(file_name, nodes=None):
    edges = pd.read_csv(csv_path(file_name), keep_default_na=False)
    G = nx.Graph()
    if nodes is not None:
        G.add_nodes_from(nodes)
    G.add_edges_from(zip(edges["source"], edges["target"]))
    return G

def load_weaver():
    # 3-column edge list: source, target, timestamp -> collapse to static undirected graph
    cached = "aves_weaver_social_edges.csv"
    if not os.path.exists(csv_path(cached)):
        zipped_folder = zipfile.ZipFile(download(WEAVER_URL, "aves-weaver-social.zip"))
        edges = pd.read_csv(
            io.BytesIO(zipped_folder.read("aves-weaver-social.edges")),
            header=None,
            sep=r"\s+", engine='python', comment="#", 
            names=["source", "target", "timestamp"]
        )
        # same pair may interact many times; keep each unique undirected edge once
        lo = edges[["source", "target"]].min(axis=1)
        hi = edges[["source", "target"]].max(axis=1)
        pairs = pd.DataFrame({"source": lo, "target": hi}) \
            .query("source != target") \
            .drop_duplicates()
        pairs.to_csv(csv_path(cached), index=False)
    return read_edges(cached)

weaver = load_weaver()
print("Weaver : ", weaver)

Weaver :  Graph with 445 nodes and 1332 edges


## 2. Community Count Estimator

...

### 2.1 Community Detection Algorithm

...

In [2]:
def construct_matrix(G: nx.Graph):
    # Fix the vertex order so matrix indices match vertices.
    nodes = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
    A_squared = A @ A
    #(A^2)_{ij} represents the number of shared neighbours between vertices i and j.
    M = np.where(A == 1,1 + A_squared, -1) #Added 1 to the entries with edges so nodes with edges 
    # Adjacency matrix squared but 0's get replaced by -1's

    np.fill_diagonal(M, -1)#Avoid self loops
    return nodes, M

def community_matching(G: nx.Graph, nodes, M, gamma = 1):

    assert 1 <= gamma <= 2
    sets = {}

    for i, v in enumerate(nodes):
        degree = G.degree(v)
        #Isolated vertices have no candidate neighbours and get assigned their own set
        if degree == 0:
            sets[v] = set()
        #added ceil function to ensure t being an integer
        t = math.ceil(gamma * math.ceil(degree / 2))
        sorted_M= np.sort(M[i])[::-1]
        
        if sorted_M[t - 1]> 0: 
            selected = {nodes[j] for j in range(len(nodes))if j != i and M[i, j] >= sorted_M[t - 1]}#Vertices above the threshold are grouped together
        else:
            selected = {nodes[j] for j in range(len(nodes)) if j != i and M[i, j] > 0}
        sets[v] = selected
    return sets



def form_communities(G, sets):
    H = nx.Graph()
    H.add_nodes_from(G.nodes())

    # Add an edge if either vertex selects the other.
    for v, selected in sets.items():
        for u in selected:
            H.add_edge(v, u)

    return [set(c) for c in nx.connected_components(H)]


#Preliminaty test against louvain
communities_louvain = nx.community.louvain_communities(weaver)
#Louvain
print("Number of communities (louvain)", len(communities_louvain))
print("Community sizes", sorted([len(c) for c in communities_louvain]))

#Implemented method
nodes, Matrix = construct_matrix(weaver)

sets = community_matching(weaver, nodes, Matrix, gamma=1)

communities = form_communities(weaver, sets)

print("Number of communities", len(communities))
print("Community sizes", sorted([len(c) for c in communities]))




Number of communities (louvain) 29
Community sizes [2, 2, 2, 3, 4, 4, 4, 4, 5, 6, 7, 9, 10, 10, 11, 11, 13, 15, 16, 18, 24, 25, 26, 27, 30, 35, 36, 41, 45]
Number of communities 31
Community sizes [2, 2, 2, 3, 4, 4, 4, 4, 4, 4, 4, 5, 6, 6, 7, 7, 7, 8, 9, 10, 11, 11, 11, 11, 14, 24, 36, 39, 43, 45, 98]


### 2.2 Network Properties Analysis

...

In [3]:
def describe_network(G, name):
    degree_list = np.array([deg for _, deg in G.degree()])
    print(f"Computing network {name}")
    return {"network": name, "n": G.number_of_nodes(), "m": G.number_of_edges(),
            "mean degree": degree_list.mean(), "max degree": int(degree_list.max()),
            "leaves": int((degree_list == 1).sum()),
            "isolated vertices": int((degree_list == 0).sum()),
            "interactions": nx.number_connected_components(G),
            "avg clustering coeff.": nx.average_clustering(G),
            "assortativity": nx.degree_assortativity_coefficient(G),
            "transitivity": nx.transitivity(G)}

display(pd.DataFrame([describe_network(weaver, "Social Weaver Network")]).set_index("network"))

Computing network Social Weaver Network


,n,m,mean degree,max degree,leaves,isolated vertices,interactions,avg clustering coeff.,assortativity,transitivity
network,,,,,,,,,,
Social Weaver Network,445,1332,5.986517,27,41,0,22,0.668505,0.199066,0.588094


### 2.3 Predictions Based on Network Properties

-> global clustering coefficient